In [ ]:
import pandas as pd
import numpy as np
import pyarrow
import random
from datetime import datetime , timedelta
import pyarrow as pa
import pyarrow.parquet as pq
import os
import duckdb

In [12]:
# --- TASK 1 --- 
# STEP 1
df_list = []
df_dict = {}
np.random.seed(42)
city_list = [ "Berlin", "Tokyo", "New York", "Baku" , "Warsaw" , "Madrid" , "Liverpool" ,"Newcastle" , "Hamburg"
            , "Munchen"]
active_list = [True, False]
for i in range(500000):
    user_id = i
    city = random.choice(city_list)
    score = np.random.randint(0,100)
    active = random.choice(active_list)
    day = random.randint(0, 1095)
    random_date = datetime.now() - timedelta(days=day, seconds=random.randint(0, 86400))
    signup_date = random_date
    age = np.random.randint(18,80)
    sessions = np.random.randint(0,500)
    revenue = np.random.randint(0,1000)
    df_dict = {"user_id" : user_id , 
               "city" : city , 
               "score" : score , 
               "active" : active , 
               "signup_date" : signup_date,
    "age" : age,
    "sessions" : sessions,
    "revenue" : revenue}
    df_list.append(df_dict)
df = pd.DataFrame(df_list)
df.head()

# Step 2: Save the dataset as a Parquet file. Then use pyarrow.parquet.ParquetFile to inspect:
#df.to_parquet('data.parquet', engine='pyarrow')

pf = pq.ParquetFile('data.parquet')

print("The number of row groups:" , pf.num_row_groups)
print(f"number of rows: {pf.metadata.num_rows}, number of columns:{len(pf.schema.names)}")
print("schema:", pf.schema)
row_group_meta = pf.metadata.row_group(0)
for i in range(row_group_meta.num_columns):
    col_meta = row_group_meta.column(i)
    stats = col_meta.statistics
    col_name = pf.schema.names[i]
    phys_type = col_meta.physical_type
    comp_size = col_meta.total_compressed_size
    min_val = stats.min if stats and stats.has_min_max else "N/A"
    max_val = stats.max if stats and stats.has_min_max else "N/A"
    null_count = stats.null_count if stats and stats.has_null_count else "N/A"
print(f"column name : {col_name} , physical type : {phys_type}, compressed size : {comp_size}, minimum value : {min_val}, maximum value : {max_val}")


The number of row groups: 1
number of rows: 500000, number of columns:8
schema: <pyarrow._parquet.ParquetSchema object at 0x000002050743FC00>
required group field_id=-1 schema {
  optional int64 field_id=-1 user_id;
  optional binary field_id=-1 city (String);
  optional int64 field_id=-1 score;
  optional boolean field_id=-1 active;
  optional int64 field_id=-1 signup_date (Timestamp(isAdjustedToUTC=false, timeUnit=nanoseconds, is_from_converted_type=false, force_set_converted_type=false));
  optional int64 field_id=-1 age;
  optional int64 field_id=-1 sessions;
  optional int64 field_id=-1 revenue;
}

column name : revenue , physical type : INT64, compressed size : 630125, minimum value : 0, maximum value : 999


In [15]:
# Step 3: Save the same dataset as CSV. Compare file sizes. Print both sizes in KB and the compression ratio.

df.to_csv('data.csv')
parquet_size = os.path.getsize('data.parquet')
csv_size = os.path.getsize('data.csv')

parquet_kb = parquet_size / 1024
csv_kb = csv_size / 1024

compression_ratio = csv_size / parquet_size

print(f"Parquet size: {parquet_kb:.2f} KB")
print(f"CSV size:     {csv_kb:.2f} KB")
print(f"compression ratio (CSV/Parquet): {compression_ratio:.2f}x")

Parquet size: 8680.66 KB
CSV size:     33361.29 KB
compression ratio (CSV/Parquet): 3.84x


#Step 4: Write a markdown cell explaining what information the Parquet metadata provides that CSV does not, 
and why that matters for query performance.

Why Parquet Metadata is Critical for Query Performance
CSV and Parquet differ fundamentally in how they store data and how much "prior knowledge" they provide to a system. CSV files are simple, row-oriented text files; they contain no structural information about their content. In contrast, Parquet includes a rich metadata layer within the file "footer."

In [35]:
# Task 2

#Step 1: Read the full Parquet file from Task 1 and time it.
%time
print(pd.read_parquet(r"data.parquet"))


# Step 2: Read only 2 columns from the same Parquet file and time it.
df_parquet = pd.read_parquet(r"data.parquet")
cols_to_read = ['user_id', 'city'] 
print("Reading only 2 columns from disk:")
%time table = pq.read_table('data.parquet', columns=cols_to_read)
print(table)


#Step 3: Save the same dataset as CSV. Compare file sizes. Print both sizes in KB and the compression ratio.

df_csv = pd.read_csv(r"data.csv")
cols_to_read = ['user_id', 'city'] 
print("Reading only 2 columns from disk:")
%time df_subset = pd.read_csv('data.csv',usecols=['user_id', 'city'])
print(df_subset)



CPU times: total: 0 ns
Wall time: 3.81 μs
        user_id       city  score  active                signup_date  age  \
0             0     Berlin     51    True 2023-03-06 20:05:56.923994   46   
1             1  Liverpool     71   False 2023-09-07 06:45:54.924038   78   
2             2  Newcastle     82   False 2024-03-15 20:25:20.924068   40   
3             3     Madrid     87    True 2024-08-29 02:49:51.924095   70   
4             4     Madrid     23   False 2023-12-27 07:43:33.924120   20   
...         ...        ...    ...     ...                        ...  ...   
499995   499995       Baku     71   False 2025-08-21 00:50:09.671020   74   
499996   499996     Warsaw     41    True 2024-12-11 06:34:57.671043   48   
499997   499997       Baku      9    True 2024-01-31 16:31:44.671067   34   
499998   499998  Newcastle     85    True 2023-12-29 14:58:56.671090   43   
499999   499999     Madrid     99    True 2024-01-17 16:40:28.671113   22   

        sessions  revenue  
0    

# Step 4: Write a markdown cell explaining why Parquet selective reads are faster. Connect your explanation to the column-chunk layout you observed in Task 1.

Parquet selective reads are significantly faster than CSV reads because Parquet is a columnar storage format, whereas CSV is a row-based format. This fundamental structural difference changes how data is accessed on disk.

1. Columnar Layout vs. Row-based Layout
Columnar Storage (Parquet): Data for each column is stored in contiguous blocks (column chunks) . Because all values for a single column are physically stored together on disk, a query requesting only specific columns can simply "jump" to the corresponding file offset and read only that data, ignoring everything else.

Row-based Storage (CSV): Data is stored line-by-line, interleaving all columns. To retrieve even a single value, the system must scan through the entire file, loading every single column into memory just to discard the parts you didn't ask for.

In [40]:
# Task 3
# Step 1: Using PyArrow, read the Parquet file with a filter (e.g., age > 50). Time the read.

%time print(duckdb.sql("SELECT * FROM 'data.parquet' WHERE age > 50"))


#Step 2: Read the full Parquet file without a filter and apply the same filter in pandas after loading. Time this approach.
%time print(pq.read_table('data.parquet'))
%time print(duckdb.sql("SELECT * FROM 'data.csv' WHERE age > 50"))

┌─────────┬───────────┬───────┬─────────┬────────────────────────────┬───────┬──────────┬─────────┐
│ user_id │   city    │ score │ active  │        signup_date         │  age  │ sessions │ revenue │
│  int64  │  varchar  │ int64 │ boolean │        timestamp_ns        │ int64 │  int64   │  int64  │
├─────────┼───────────┼───────┼─────────┼────────────────────────────┼───────┼──────────┼─────────┤
│       1 │ Liverpool │    71 │ false   │ 2023-09-07 06:45:54.924038 │    78 │       20 │     614 │
│       3 │ Madrid    │    87 │ true    │ 2024-08-29 02:49:51.924095 │    70 │       99 │     871 │
│       8 │ Tokyo     │    21 │ true    │ 2025-05-28 23:35:01.924221 │    78 │      235 │     856 │
│      11 │ Hamburg   │    14 │ true    │ 2024-06-12 20:19:48.924295 │    79 │      445 │     686 │
│      12 │ Munchen   │    61 │ true    │ 2024-06-03 02:58:27.92432  │    68 │      363 │     566 │
│      13 │ New York  │    63 │ true    │ 2024-10-02 16:04:27.924345 │    74 │      130 │     484 │


# Step 3: Compare the number of rows returned and the execution times. Write a markdown cell explaining what predicate pushdown does and why it is faster.


Predicate Pushdown is an optimization technique where the query filter (the "predicate," such as WHERE age > 50) is "pushed down" from the application layer directly to the storage layer .

In [42]:
# Step 4: Run the same filtered query using DuckDB's SQL interface directly on the Parquet file:

%time print(duckdb.sql("SELECT * FROM 'data.parquet' WHERE age > 50"))

┌─────────┬───────────┬───────┬─────────┬────────────────────────────┬───────┬──────────┬─────────┐
│ user_id │   city    │ score │ active  │        signup_date         │  age  │ sessions │ revenue │
│  int64  │  varchar  │ int64 │ boolean │        timestamp_ns        │ int64 │  int64   │  int64  │
├─────────┼───────────┼───────┼─────────┼────────────────────────────┼───────┼──────────┼─────────┤
│       1 │ Liverpool │    71 │ false   │ 2023-09-07 06:45:54.924038 │    78 │       20 │     614 │
│       3 │ Madrid    │    87 │ true    │ 2024-08-29 02:49:51.924095 │    70 │       99 │     871 │
│       8 │ Tokyo     │    21 │ true    │ 2025-05-28 23:35:01.924221 │    78 │      235 │     856 │
│      11 │ Hamburg   │    14 │ true    │ 2024-06-12 20:19:48.924295 │    79 │      445 │     686 │
│      12 │ Munchen   │    61 │ true    │ 2024-06-03 02:58:27.92432  │    68 │      363 │     566 │
│      13 │ New York  │    63 │ true    │ 2024-10-02 16:04:27.924345 │    74 │      130 │     484 │


In [119]:
# Task 4

# Query 1: Count records per city.
%time print("using Pandas:", df.groupby("city").size())
%time print("using DuckDB:", duckdb.sql("SELECT city, COUNT(*) FROM 'data.parquet' GROUP BY city"))

# Query 2: Average score by city, ordered by average score descending.
%time print("using Pandas:", df.groupby("city")["score"].mean().sort_values(ascending=False))
%time print("using DuckDB:", duckdb.sql("SELECT city, AVG(score) as AVG_SCORE FROM 'data.parquet' GROUP BY city ORDER BY AVG_SCORE DESC"))

# Query 3: For each city, find the percentage of active users whose score is above 75.
%time print("using Pandas:" , len(df[df["score"] > 75]) / len(df))
%time print("using DuckDB:" , duckdb.sql("SELECT COUNT(*) FILTER(WHERE score > 75) / COUNT(*)::FLOAT FROM 'data.parquet'").fetchone()[0])

# Query 4: Find the top 10 users by score for each city (window function / rank).
%time print("using Pandas:" , df.groupby('city').apply(lambda x: x.nlargest(10, 'score')).reset_index(drop=True))
%time print("using DuckDB:" , duckdb.sql("SELECT * FROM ( SELECT *, ROW_NUMBER() OVER (PARTITION BY city ORDER BY score DESC) as rank FROM 'data.parquet') WHERE rank <= 10"))

# Query 5: Compute a running total of scores ordered by user_id, partitioned by city.
%time print("using Pandas:" ,df.sort_values(by=["city", "user_id"]).assign(running_total=lambda x: x.groupby("city")["score"].cumsum()))
%time print("using DuckDB:" , duckdb.sql("SELECT city, user_id, score, SUM(score) OVER (PARTITION BY city ORDER BY user_id) as running_total FROM 'data.parquet'"))




using Pandas: city
Baku         49635
Berlin       50142
Hamburg      50035
Liverpool    49639
Madrid       50326
Munchen      49949
New York     50281
Newcastle    50033
Tokyo        49690
Warsaw       50270
dtype: int64
CPU times: total: 31.2 ms
Wall time: 26.4 ms
using DuckDB: ┌───────────┬──────────────┐
│   city    │ count_star() │
│  varchar  │    int64     │
├───────────┼──────────────┤
│ Tokyo     │        50034 │
│ New York  │        49850 │
│ Liverpool │        50193 │
│ Hamburg   │        49836 │
│ Warsaw    │        50050 │
│ Madrid    │        50172 │
│ Munchen   │        49953 │
│ Baku      │        49982 │
│ Berlin    │        49807 │
│ Newcastle │        50123 │
├───────────┴──────────────┤
│ 10 rows        2 columns │
└──────────────────────────┘

CPU times: total: 0 ns
Wall time: 4.35 ms
using Pandas: city
Hamburg      49.647187
Newcastle    49.626806
Warsaw       49.531570
Berlin       49.518248
Tokyo        49.515878
Madrid       49.470433
New York     49.453372
Bak

<timed eval>:1: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.


┌─────────┬─────────┬───────┬─────────┬────────────────────────────┬───────┬──────────┬─────────┬───────┐
│ user_id │  city   │ score │ active  │        signup_date         │  age  │ sessions │ revenue │ rank  │
│  int64  │ varchar │ int64 │ boolean │        timestamp_ns        │ int64 │  int64   │  int64  │ int64 │
├─────────┼─────────┼───────┼─────────┼────────────────────────────┼───────┼──────────┼─────────┼───────┤
│  354610 │ Madrid  │    99 │ false   │ 2025-11-20 15:23:19.257545 │    22 │      244 │     917 │     1 │
│  441065 │ Madrid  │    99 │ true    │ 2023-05-10 11:45:24.285832 │    55 │       61 │     770 │     2 │
│   43155 │ Madrid  │    99 │ false   │ 2024-02-20 15:29:42.941408 │    47 │      143 │     392 │     3 │
│  310716 │ Madrid  │    99 │ false   │ 2023-09-04 10:28:33.227312 │    43 │      237 │     122 │     4 │
│  287146 │ Madrid  │    99 │ true    │ 2023-07-28 14:45:19.673141 │    26 │      268 │     550 │     5 │
│  252688 │ Madrid  │    99 │ false   │ 2024-1

Step 5: Write a markdown cell comparing:

Which tool was easier to express each query in?
sql was much more easier tool for filtering values

Which was faster?
pandas

For which queries did the difference matter most?
espicially sort_values and groupby

In [142]:
# --- TASK 5 --- 
# Step 1: Create a pandas DataFrame and convert it to an Arrow Table using pyarrow.Table.from_pandas().
df_list = []
df_dict = {}
np.random.seed(42)
city_list = [ "Berlin", "Tokyo", "New York", "Baku" , "Warsaw" , "Madrid" , "Liverpool" ,"Newcastle" , "Hamburg"
            , "Munchen"]
active_list = [True, False]
for i in range(500000):
    user_id = i
    city = random.choice(city_list)
    score = np.random.randint(0,100)
    active = random.choice(active_list)
    day = random.randint(0, 1095)
    random_date = datetime.now() - timedelta(days=day, seconds=random.randint(0, 86400))
    signup_date = random_date
    age = np.random.randint(18,80)
    sessions = np.random.randint(0,500)
    revenue = np.random.randint(0,1000)
    df_dict = {"user_id" : user_id , 
               "city" : city , 
               "score" : score , 
               "active" : active , 
               "signup_date" : signup_date,
    "age" : age,
    "sessions" : sessions,
    "revenue" : revenue}
    df_list.append(df_dict)
df = pd.DataFrame(df_list)

table = pa.Table.from_pandas(df)

# Step 2: Inspect the Arrow Table schema and compare it to the pandas dtypes. Note any differences.

print(table.schema == df.dtypes)

# Step 3: Write the Arrow Table to Parquet using pyarrow.parquet.write_table(). Read it back into a new Arrow Table.
pq.write_table(table, 'output_data.parquet')
print(pq.read_table('output_data.parquet'))

# Step 4: Convert the Arrow Table back to a pandas DataFrame using .to_pandas(). Verify the data matches the original.
a = pq.read_table('output_data.parquet')
df_ = a.to_pandas()
print(df_.shape == a.shape)

# Step 5: Demonstrate the pandas dtype_backend="pyarrow" option by reading the Parquet file with Arrow-backed dtypes. Print the dtypes and compare with traditional pandas dtypes.
df_numpy = pd.read_parquet('data.parquet')
df_arrow = pd.read_parquet('data.parquet', dtype_backend="pyarrow")
print(df_numpy.dtypes)
print(df_arrow.dtypes)



user_id        False
city           False
score          False
active         False
signup_date    False
age            False
sessions       False
revenue        False
dtype: bool
pyarrow.Table
user_id: int64
city: string
score: int64
active: bool
signup_date: timestamp[ns]
age: int64
sessions: int64
revenue: int64
----
user_id: [[0,1,2,3,4,...,131067,131068,131069,131070,131071],[131072,131073,131074,131075,131076,...,262139,262140,262141,262142,262143],[262144,262145,262146,262147,262148,...,393211,393212,393213,393214,393215],[393216,393217,393218,393219,393220,...,499995,499996,499997,499998,499999]]
city: [["Madrid","Madrid","Tokyo","Munchen","Newcastle",...,"Madrid","Newcastle","Baku","Berlin","Hamburg"],["Tokyo","Liverpool","New York","Hamburg","Tokyo",...,"Baku","Berlin","Warsaw","Baku","Newcastle"],["Madrid","Newcastle","Warsaw","Warsaw","Newcastle",...,"Madrid","Warsaw","Baku","Hamburg","Newcastle"],["Madrid","New York","Newcastle","Warsaw","Newcastle",...,"Munchen","Munchen"

Step 6: Write a markdown cell explaining the role of Arrow in the modern analytics stack. How does it connect Parquet (disk) to pandas (analysis) to DuckDB (SQL)?

Apache Arrow has become the "common language" of modern data analytics. It serves as a high-performance, language-agnostic, columnar memory format that allows different tools—like Parquet, Pandas, and DuckDB—to communicate without the overhead of copying or converting data.

1. From Disk to Memory: Parquet and Arrow
Parquet is the industry standard for on-disk storage because it is highly compressed and optimized for columnar access. When you read a Parquet file, Apache Arrow acts as the bridge. Instead of serializing data into slow, row-based formats (like standard Python lists), Arrow maps the Parquet columns directly into memory. This allows for extremely fast loading times because the data format on disk is structurally aligned with the format in RAM.

2. Analysis and Manipulation: Pandas and Arrow
Traditionally, Pandas relied on NumPy for data storage, which was never designed for modern data challenges like handling large strings or complex null/missing values. With the dtype_backend="pyarrow" option, Pandas uses Arrow arrays as its internal storage. This eliminates the need for expensive "object" type conversions. As a result, operations are performed in native C++ code rather than Python loops, significantly boosting performance.

3. High-Performance SQL: DuckDB and Arrow
DuckDB is an in-process OLAP database designed for analytic queries. Because DuckDB and Arrow both use a columnar memory layout, they can perform "zero-copy" data exchange. When you pass a DataFrame from Pandas to DuckDB, the data isn't physically moved or duplicated. DuckDB simply points to the memory addresses already occupied by the Arrow data. This efficiency allows SQL queries to run on dataframes with virtually no latency penalty.